# TF-IDF training — side classification (left / center / right)

Main baseline for the project (stronger than frozen HerBERT on this dataset).

This notebook:
1. loads splits from `data/term9/splits/`
2. trains **word + char TF-IDF** + balanced Logistic Regression
3. calibrates the **left** class bias on validation
4. plots class distributions, confusion matrices, and F1 scores
5. saves the model to `models/`

Run with the project kernel: **Python (data-mining-project)** / `.venv`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    recall_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import FeatureUnion, Pipeline

# Prefer running from the repo root (where pyproject.toml lives).
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    for parent in [ROOT, *ROOT.parents]:
        if (parent / "pyproject.toml").exists():
            ROOT = parent
            break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from data_mining_project.utils.sejm_text import SEJM_STOP_WORDS, clean_speech_text
from data_mining_project.utils.side_calibration import (
    predict_with_left_bias,
    tune_left_bias,
)

SPLITS_DIR = ROOT / "data" / "term9" / "splits"
MODELS_DIR = ROOT / "models"
SIDE_ORDER = ["left", "center", "right"]

print("ROOT:", ROOT)
print("splits exist:", (SPLITS_DIR / "train.csv").exists())

## 1. Data and class distribution

In [ ]:
splits = {
    name: pd.read_csv(SPLITS_DIR / f"{name}.csv")
    for name in ("train", "val", "test")
}

for name, df in splits.items():
    print(f"{name}: {len(df):,} speeches | members={df['member_id'].nunique()}")
    print(df["side"].value_counts().reindex(SIDE_ORDER).to_string())
    print()

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, name in zip(axes, ("train", "val", "test")):
    counts = splits[name]["side"].value_counts().reindex(SIDE_ORDER)
    ax.bar(counts.index, counts.values, color=["#c44e52", "#4c72b0", "#55a868"])
    ax.set_title(name)
    ax.set_xlabel("side")
axes[0].set_ylabel("number of speeches")
fig.suptitle("Side class distribution across splits", y=1.02)
fig.tight_layout()
plt.show()

## 2. Build TF-IDF + Logistic Regression pipeline

Word n-grams (1–2) + character n-grams (3–5), with Sejm procedural stop-words removed.

In [ ]:
def build_side_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            (
                "features",
                FeatureUnion(
                    transformer_list=[
                        (
                            "word",
                            TfidfVectorizer(
                                lowercase=True,
                                analyzer="word",
                                ngram_range=(1, 2),
                                min_df=3,
                                max_df=0.9,
                                max_features=80_000,
                                sublinear_tf=True,
                                stop_words=sorted(SEJM_STOP_WORDS),
                            ),
                        ),
                        (
                            "char",
                            TfidfVectorizer(
                                lowercase=True,
                                analyzer="char_wb",
                                ngram_range=(3, 5),
                                min_df=3,
                                max_features=40_000,
                                sublinear_tf=True,
                            ),
                        ),
                    ]
                ),
            ),
            (
                "clf",
                OneVsRestClassifier(
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                        solver="liblinear",
                    )
                ),
            ),
        ]
    )


pipe = build_side_pipeline()
print("Fitting on", f"{len(splits['train']):,}", "train speeches...")
pipe.fit(splits["train"]["text"], splits["train"]["side"])
print("classes:", list(pipe.classes_))

## 3. Calibrate left-class bias (on val)

Search for a positive bias on the `left` class probability that maximizes **left F1** on validation.

In [ ]:
biases = np.linspace(0.0, 0.45, 46)
rows = []
for bias in biases:
    y_pred = predict_with_left_bias(pipe, splits["val"]["text"], float(bias))
    left_f1 = f1_score(
        splits["val"]["side"],
        y_pred,
        labels=["left"],
        average="macro",
        zero_division=0,
    )
    macro_f1 = f1_score(
        splits["val"]["side"], y_pred, average="macro", zero_division=0
    )
    left_recall = recall_score(
        splits["val"]["side"],
        y_pred,
        labels=["left"],
        average="macro",
        zero_division=0,
    )
    rows.append(
        {
            "bias": float(bias),
            "left_f1": float(left_f1),
            "macro_f1": float(macro_f1),
            "left_recall": float(left_recall),
        }
    )

curve = pd.DataFrame(rows)
left_bias, best_left_f1, best_macro_f1 = tune_left_bias(
    pipe, splits["val"]["text"], splits["val"]["side"], biases=biases
)
print(
    f"Selected left_bias={left_bias:.3f} | val left_f1={best_left_f1:.3f} | "
    f"val macro_f1={best_macro_f1:.3f}"
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve["bias"], curve["left_f1"], label="left F1")
ax.plot(curve["bias"], curve["macro_f1"], label="macro F1")
ax.plot(curve["bias"], curve["left_recall"], label="left recall", linestyle="--")
ax.axvline(left_bias, color="black", linestyle=":", label=f"selected bias={left_bias:.3f}")
ax.set_xlabel("left bias")
ax.set_ylabel("score")
ax.set_title("Left bias calibration on validation (TF-IDF)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 4. Evaluation: raw vs calibrated

In [ ]:
def evaluate(name: str, y_true, y_pred):
    report = classification_report(y_true, y_pred, digits=3, zero_division=0)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    acc = float((np.asarray(y_pred) == np.asarray(y_true)).mean())
    print(f"=== {name} | acc={acc:.3f} macro_f1={macro:.3f} ===")
    print(report)
    return {
        "accuracy": acc,
        "macro_f1": float(macro),
        "report": classification_report(
            y_true, y_pred, digits=3, zero_division=0, output_dict=True
        ),
    }


results = {}
for split_name in ("val", "test"):
    y_true = splits[split_name]["side"]
    texts = splits[split_name]["text"]
    y_raw = pipe.predict(texts)
    y_cal = predict_with_left_bias(pipe, texts, left_bias)
    results[split_name] = {
        "raw": evaluate(f"{split_name} RAW", y_true, y_raw),
        "calibrated": evaluate(f"{split_name} CALIBRATED", y_true, y_cal),
        "y_true": y_true,
        "y_raw": y_raw,
        "y_cal": y_cal,
    }

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, kind, title in zip(
    axes,
    ("y_raw", "y_cal"),
    ("Test — no calibration", f"Test — left_bias={left_bias:.3f}"),
):
    ConfusionMatrixDisplay.from_predictions(
        results["test"]["y_true"],
        results["test"][kind],
        labels=SIDE_ORDER,
        ax=ax,
        colorbar=False,
    )
    ax.set_title(title)
fig.suptitle("Confusion matrix (test) — TF-IDF", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
def f1_per_class(report_dict):
    return [report_dict[label]["f1-score"] for label in SIDE_ORDER]


x = np.arange(len(SIDE_ORDER))
width = 0.35
raw_f1 = f1_per_class(results["test"]["raw"]["report"])
cal_f1 = f1_per_class(results["test"]["calibrated"]["report"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width / 2, raw_f1, width, label="raw")
ax.bar(x + width / 2, cal_f1, width, label="calibrated")
ax.set_xticks(x)
ax.set_xticklabels(SIDE_ORDER)
ax.set_ylim(0, 1)
ax.set_ylabel("F1")
ax.set_title("Test F1 per class: raw vs calibrated (TF-IDF)")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
for i, (a, b) in enumerate(zip(raw_f1, cal_f1)):
    ax.text(i - width / 2, a + 0.02, f"{a:.2f}", ha="center", fontsize=9)
    ax.text(i + width / 2, b + 0.02, f"{b:.2f}", ha="center", fontsize=9)
plt.show()

print(
    "left F1 test: "
    f"raw={raw_f1[0]:.3f} -> calibrated={cal_f1[0]:.3f} "
    f"(delta={cal_f1[0] - raw_f1[0]:+.3f})"
)
print(
    "macro F1 test: "
    f"raw={results['test']['raw']['macro_f1']:.3f} -> "
    f"calibrated={results['test']['calibrated']['macro_f1']:.3f}"
)

## 5. Save model

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODELS_DIR / "side_tfidf.joblib"
calib_path = MODELS_DIR / "side_tfidf_calibration.json"
metrics_path = MODELS_DIR / "tfidf_metrics.json"

joblib.dump(pipe, model_path)
calib_path.write_text(
    json.dumps(
        {
            "left_bias": left_bias,
            "left_label": "left",
            "model": "tfidf",
        },
        indent=2,
    ),
    encoding="utf-8",
)

metrics = {
    "model": "tfidf_word_char",
    "side": {
        "left_bias": left_bias,
        "calibration": {
            "val_left_f1": round(best_left_f1, 4),
            "val_macro_f1": round(best_macro_f1, 4),
        },
        "val": {
            "raw": results["val"]["raw"],
            "calibrated": results["val"]["calibrated"],
        },
        "test": {
            "raw": results["test"]["raw"],
            "calibrated": results["test"]["calibrated"],
        },
    },
}
metrics_path.write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False, default=float),
    encoding="utf-8",
)

print("Saved:")
print(" -", model_path)
print(" -", calib_path)
print(" -", metrics_path)

## 6. Quick predict demo

After saving, you can also use: `uv run predict-side "..."` (if CLI points at the TF-IDF model).

In [ ]:
demo_texts = [
    "Musimy wzmocnić państwo opiekuńcze, podnieść płacę minimalną i chronić prawa pracownicze.",
    "Potrzebujemy kompromisu w sprawie praworządności i współpracy z Unią Europejską.",
    "Należy obniżyć podatki, ograniczyć biurokrację i bronić tradycyjnych wartości.",
]
demo_clean = [clean_speech_text(t) or t for t in demo_texts]

demo_pred = predict_with_left_bias(pipe, demo_clean, left_bias)
demo_proba = pipe.predict_proba(demo_clean)

for text, label, proba in zip(demo_texts, demo_pred, demo_proba):
    ranked = sorted(zip(pipe.classes_, proba), key=lambda x: x[1], reverse=True)
    print(f"[{label}] {text}")
    print("   ", ", ".join(f"{c}={p:.2f}" for c, p in ranked))
    print()